# NFIP Insurance Analytics

End-to-end analysis of the seven Gold layer KPI views built on 2.7 million NFIP claims and policy records across FL, LA, TX, NJ, and NY.

**Prerequisites**
- Docker container running: `docker-compose up -d`
- Full pipeline executed: `./scripts/run_all_sql.sh`
- Dependencies installed: `pip install -r requirements.txt`

In [ ]:
import pyodbc
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

%matplotlib inline

# Clean white grid background for all charts
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

# Consistent colour palette used across all charts
BLUE   = "#2171b5"
ORANGE = "#e6550d"
GREEN  = "#31a354"
RED    = "#de2d26"

## Connect to the Database

The pipeline runs inside an Azure SQL Edge Docker container on port 1433.

In [ ]:
# Detect which ODBC driver version is installed (18 preferred, 17 as fallback)
driver = next(
    (d for d in ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server"]
     if d in pyodbc.drivers()),
    None
)

# Connect to the local Docker container
conn = pyodbc.connect(
    f"Driver={{{driver}}};Server=localhost,1433;"
    "Database=NfipInsuranceWarehouse;UID=sa;"
    "PWD=NfipWarehouse2026!;TrustServerCertificate=yes"
)

print("Connected to NfipInsuranceWarehouse ✓")

## 1. Loss Ratio by State and Year

**Loss ratio = claims paid ÷ premium collected.** A value above 1.0 means the insurer paid out more than it collected. The heatmap makes it immediately obvious which states and years were catastrophic.

*Source view: `gold.vw_loss_ratio_by_state`*

In [ ]:
# Query one row per state-year — filtered to the 2009-2024 policy period
df = pd.read_sql("""
    SELECT state_name, year, loss_ratio
    FROM gold.vw_loss_ratio_by_state
    WHERE year BETWEEN 2009 AND 2024
      AND total_premium > 0
      AND loss_ratio IS NOT NULL
""", conn)

# Reshape to a state x year matrix so the heatmap has states as rows
pivot = df.pivot_table(index="state_name", columns="year", values="loss_ratio")

fig, ax = plt.subplots(figsize=(14, 4))

# RdYlGn_r: red = high loss ratio (bad), green = low (profitable)
# center=1.0 anchors the colour split at the breakeven point
sns.heatmap(
    pivot, annot=True, fmt=".2f",
    cmap="RdYlGn_r", center=1.0, vmin=0, vmax=3,
    linewidths=0.5, linecolor="#cccccc",
    ax=ax, cbar_kws={"label": "Loss Ratio"}
)

ax.set_title(
    "Loss Ratio by State and Year (2009–2024)\n"
    "Red = claims exceed premiums  |  Green = profitable",
    pad=12, fontsize=13
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()

## 2. Claims Development by Accident Year

Two things matter for any loss year: **how many claims** were filed, and **how large** each one was. The dual-axis chart shows total paid (bars, left) alongside average severity per claim (line, right). Katrina 2005 and Harvey 2017 are annotated — they define the worst-case scenarios for NFIP.

*Source view: `gold.vw_claims_development`*

In [ ]:
# Year of loss = when the flood occurred, not when the claim was filed
df = pd.read_sql("""
    SELECT year_of_loss, claim_count, total_paid, avg_paid_per_claim
    FROM gold.vw_claims_development
    WHERE year_of_loss BETWEEN 1978 AND 2024
    ORDER BY year_of_loss
""", conn)

# Scale to billions so the axis labels stay readable
df["total_paid_B"] = df["total_paid"] / 1e9

fig, ax1 = plt.subplots(figsize=(14, 5))

# Left axis: bars for total paid per accident year
ax1.bar(df["year_of_loss"], df["total_paid_B"], color=BLUE, alpha=0.8, label="Total Paid ($B)")
ax1.set_ylabel("Total Paid ($B)", color=BLUE)
ax1.tick_params(axis="y", labelcolor=BLUE)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}B"))

# Right axis: line for average severity per claim
ax2 = ax1.twinx()
ax2.plot(df["year_of_loss"], df["avg_paid_per_claim"] / 1000,
         color=ORANGE, linewidth=2, label="Avg Severity ($K)")
ax2.set_ylabel("Avg Severity ($K)", color=ORANGE)
ax2.tick_params(axis="y", labelcolor=ORANGE)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.0f}K"))

# Annotate the two dominant catastrophe years
for yr, label in [(2005, "Katrina\n2005"), (2017, "Harvey\n2017")]:
    row = df[df["year_of_loss"] == yr]
    if not row.empty:
        y_val = row["total_paid_B"].values[0]
        ax1.annotate(label, xy=(yr, y_val), xytext=(yr + 1, y_val + 0.8),
                     ha="left", fontsize=8, color="#333333",
                     arrowprops=dict(arrowstyle="->", color="#888888", lw=0.8))

ax1.set_title("Claims Development by Accident Year (1978–2024)", pad=12, fontsize=13)
ax1.set_xlabel("Year of Loss")

# Merge both axes legends into one box
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()

## 3. Large Loss Concentration by State

Claims above the **95th percentile ($185,607)** are classified as catastrophic. This chart shows what share of all claims dollars came from those tail events, by state. The skew toward Florida and Texas reflects hurricane and storm-surge exposure — the same concentration that drives reinsurance design.

*Source view: `gold.vw_large_loss_concentration`*

In [ ]:
# The P95 threshold is pre-calculated in the view — one row per state
df = pd.read_sql("""
    SELECT state_name, pct_of_total_paid, large_loss_count, large_loss_total
    FROM gold.vw_large_loss_concentration
    ORDER BY pct_of_total_paid DESC
""", conn)

fig, ax = plt.subplots(figsize=(8, 5))

# Horizontal bars make the state names easier to read
bars = ax.barh(df["state_name"], df["pct_of_total_paid"], color=BLUE, alpha=0.85)

# Add percentage labels to the right of each bar
for bar, val in zip(bars, df["pct_of_total_paid"]):
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=10)

ax.set_xlabel("% of Total Claims Paid from P95+ Losses")
ax.set_title(
    "Large Loss Concentration by State\n"
    "Share of total paid from catastrophic claims  |  P95 threshold: $185,607",
    pad=12, fontsize=13
)
ax.set_xlim(0, df["pct_of_total_paid"].max() * 1.18)
ax.invert_yaxis()  # Highest percentage at top
plt.tight_layout()

## 4. Severity by Flood Zone

Each zone category carries a different loss profile. Zone V (coastal, storm surge) losses are heavily building-damage driven. The 19% gap between Zone A and Zone X is the actuarial basis for SFHA mandatory purchase requirements.

*Source view: `gold.vw_severity_by_flood_zone`*

In [ ]:
df = pd.read_sql("""
    SELECT zone_category, avg_severity, avg_building_paid, avg_contents_paid
    FROM gold.vw_severity_by_flood_zone
    ORDER BY avg_severity DESC
""", conn)

x = list(range(len(df)))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

# Side-by-side bars split the loss between building and contents damage
ax.bar([i - width / 2 for i in x], df["avg_building_paid"], width,
       label="Avg Building Paid", color=BLUE, alpha=0.85)
ax.bar([i + width / 2 for i in x], df["avg_contents_paid"], width,
       label="Avg Contents Paid", color=ORANGE, alpha=0.85)

# Diamond line shows the combined total severity as a reference
ax.plot(x, df["avg_severity"], "D-", color="#333333",
        linewidth=2, markersize=7, label="Avg Total Severity", zorder=5)

ax.set_xticks(x)
ax.set_xticklabels(df["zone_category"])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_ylabel("Average Paid per Claim ($)")
ax.set_title("Average Claim Severity by Flood Zone Category\nBuilding vs. contents split",
             pad=12, fontsize=13)
ax.legend()
plt.tight_layout()

## 5. Portfolio Summary (2009–2024)

A year-by-year view of the entire book. The **dashed red line marks 1.0** — the breakeven point. Years above the line mean NFIP paid out more than it collected, requiring Treasury borrowing to cover the shortfall.

*Source view: `gold.vw_portfolio_summary`*

In [ ]:
df = pd.read_sql("""
    SELECT year, total_claims_paid, total_premium, loss_ratio, claim_count
    FROM gold.vw_portfolio_summary
    WHERE year BETWEEN 2009 AND 2024
      AND total_premium > 0
      AND loss_ratio IS NOT NULL
    ORDER BY year
""", conn)

# Scale to billions for the bar axis
df["total_claims_paid_B"] = df["total_claims_paid"] / 1e9

fig, ax1 = plt.subplots(figsize=(13, 5))

# Left axis: total claims paid per year
ax1.bar(df["year"], df["total_claims_paid_B"], color=BLUE, alpha=0.7, label="Total Claims Paid ($B)")
ax1.set_ylabel("Total Claims Paid ($B)", color=BLUE)
ax1.tick_params(axis="y", labelcolor=BLUE)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}B"))

# Right axis: loss ratio trend with breakeven reference line
ax2 = ax1.twinx()
ax2.plot(df["year"], df["loss_ratio"], color=RED, linewidth=2.5,
         marker="o", markersize=5, label="Loss Ratio")
ax2.axhline(1.0, color=RED, linestyle="--", linewidth=1, alpha=0.5)  # breakeven
ax2.set_ylabel("Loss Ratio", color=RED)
ax2.tick_params(axis="y", labelcolor=RED)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}x"))

ax1.set_title(
    "Portfolio Loss Ratio and Claims Paid (2009–2024)\n"
    "Dashed line = loss ratio of 1.0 (breakeven)",
    pad=12, fontsize=13
)
ax1.set_xlabel("Year")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()

## 6. Claims Frequency vs. Severity by Flood Zone

Expected loss = **frequency × severity**. This scatter separates those two components by zone. Zones in the top-right are the most expensive to insure. Bubble size reflects total claim volume.

*Source view: `gold.vw_claims_frequency_severity`*

In [ ]:
# Aggregate across all years — one bubble per flood zone
df = pd.read_sql("""
    SELECT
        zone_category,
        SUM(claim_count)    AS total_claims,
        SUM(total_exposure) AS total_exposure,
        SUM(total_paid) / NULLIF(SUM(claim_count), 0) AS avg_severity
    FROM gold.vw_claims_frequency_severity
    WHERE total_exposure > 0
    GROUP BY zone_category
""", conn)

# Frequency = how many claims per unit of earned exposure
df["frequency"] = df["total_claims"] / df["total_exposure"]
df = df.dropna(subset=["frequency", "avg_severity"])

fig, ax = plt.subplots(figsize=(9, 6))
colors = [BLUE, ORANGE, GREEN, RED, "#756bb1", "#636363"]

for i, (_, row) in enumerate(df.iterrows()):
    # Bubble size scales with claim count so bigger zones appear larger
    ax.scatter(
        row["frequency"], row["avg_severity"] / 1000,
        s=row["total_claims"] / df["total_claims"].max() * 3000,
        color=colors[i % len(colors)], alpha=0.75,
        edgecolors="white", linewidths=1.5
    )
    ax.annotate(row["zone_category"],
                xy=(row["frequency"], row["avg_severity"] / 1000),
                xytext=(8, 4), textcoords="offset points",
                fontsize=10, fontweight="bold")

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.3f}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}K"))
ax.set_xlabel("Claims Frequency (claims per unit of earned exposure)")
ax.set_ylabel("Average Severity ($K per claim)")
ax.set_title(
    "Claims Frequency vs. Severity by Flood Zone\nBubble size proportional to total claim count",
    pad=12, fontsize=13
)
plt.tight_layout()

## 7. Premium Adequacy by Occupancy Type

**Pure premium = total losses ÷ earned exposure** — the theoretical minimum premium needed to cover claims. Where the red bar exceeds the green, that segment is loss-making on a standalone basis.

*Source view: `gold.vw_premium_adequacy`*

In [ ]:
# One row per occupancy type, using a policy-count weighted average premium
df = pd.read_sql("""
    SELECT
        occupancy_type,
        SUM(total_claims)   AS total_claims,
        SUM(total_exposure) AS total_exposure,
        SUM(avg_premium * policy_count) / NULLIF(SUM(policy_count), 0) AS weighted_avg_premium
    FROM gold.vw_premium_adequacy
    WHERE total_exposure > 0 AND avg_premium IS NOT NULL
    GROUP BY occupancy_type
""", conn)

# Pure premium = loss cost per unit of exposure — the floor for break-even pricing
df["pure_premium"] = df["total_claims"] / df["total_exposure"]
df = df.dropna(subset=["pure_premium", "weighted_avg_premium"])
df = df.sort_values("pure_premium", ascending=False).head(10)

x = list(range(len(df)))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))

# Red = what's needed, Green = what was charged
ax.bar([i - width / 2 for i in x], df["pure_premium"], width,
       label="Pure Premium (losses ÷ exposure)", color=RED, alpha=0.8)
ax.bar([i + width / 2 for i in x], df["weighted_avg_premium"], width,
       label="Avg Premium Charged", color=GREEN, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(df["occupancy_type"], rotation=35, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_ylabel("Amount ($)")
ax.set_title(
    "Premium Adequacy by Occupancy Type\n"
    "Red = pure premium required  |  Green = avg premium charged",
    pad=12, fontsize=13
)
ax.legend()
plt.tight_layout()

In [ ]:
conn.close()
print("Connection closed.")